# Market data, descriptives, econometric tests and simulation for the temporal-rule-mining paper

Run everything with `python main.py` (or single steps: `python main.py data tables tests simulation export`).
Requires numpy, pandas, scipy, matplotlib, statsmodels and arch (`pip install -r requirements.txt`).
If simulated datasets already exist, the simulation is not regenerated unless asked (`--force-simulation`; see `simulation/README_simulation.ipynb`).

**All tests and the simulation use the same series and definitions as the rules**: gold of the rule vintage, highVol at 30% of the window, and the inflation state of the rule pipeline (centred moving average).

## Data

Yahoo Finance and FRED series retrieved on 17 October 2026; the Gürkaynak–Sack–Wright curve on 20 October 2026.
**Gold is the vintage used by the rule results**: Yahoo GC=F as downloaded by the coauthors on 10 July 2026.

| Raw file (`market_data/raw/`) | Source | Used for |
|---|---|---|
| `GSPC.csv` | Yahoo ^GSPC | S&P 500 (`SP500`); its trading days define the calendar |
| `GC_jul2026.csv` | Yahoo GC=F, downloaded 10 Jul 2026 (2005–2024) | Gold used in the paper (`GOLD_FUT`) |
| `GC.csv` | Yahoo GC=F, downloaded 17 Oct 2026 | Gold before 2005 (where the two downloads coincide) and the revised series `GOLD_FUT_REV` |
| `BTC.csv` | Yahoo BTC-USD | Bitcoin (`BTC`) |
| `feds200628.csv` | Federal Reserve, Gürkaynak–Sack–Wright (2007) | Zero-coupon Treasury returns (`ZCB*`) |
| `CPIAUCSL.csv` | FRED CPIAUCSL | US CPI and inflation states |

**Gold vintages.** Yahoo revised GC=F after July 2026: the two downloads coincide until 28 May 2020 and differ afterwards, mostly on contract-roll days (return correlation 0.9986). The rule results in the paper use the July download, so all tests and figures use it too (`GOLD_FUT`). The October download is kept as `GOLD_FUT_REV`; the econometric tests on it are in `figures_tables_repository/tab_econ_tests_gold_revised.csv`.
`GC_jul2026.csv` is `data/raw/Gold_raw.csv` from the coauthors' archive `TSLearning_raw_2026-11-11.zip` (archive SHA-256 `94c6497720cf94ca39cbbe79888812401529bb084dbe11a38eeca1410e3db2d6`; `Gold_raw.csv` SHA-256 `1cce94a826fea976fd36d984bd0113273f40697fb5d4490fab5230f7bccc0b8e`). The other series in the archive (S&P 500, Bitcoin, TLT, CPI) are identical to ours.

**Not used (`notused/`, excluded from git).** NASDAQ, IEF, TLT, the FRED constant-maturity yields (DGS2, DGS10, DGS30, and the original DGS10 download), the raw FRED dump and the coauthors' archive. The code that builds the ETF total returns, the par-bond returns and the yield changes from them is commented out in `market_data/build.py`.

## Files in `market_data/`

| File | Content |
|---|---|
| `build.py` | Builds the files below from `raw/` |
| `market_daily_levels.csv` | Daily levels on the NYSE trading calendar (days with an S&P 500 close) to 31 Dec 2024, and GSW zero-coupon yields |
| `market_daily_logreturns.csv` | Daily log returns on the same calendar: `SP500`, `GOLD_FUT`, `GOLD_FUT_REV`, `BTC`, `ZCB2Y`, `ZCB5Y`, `ZCB10Y`, `ZCB30Y` |
| `us_cpi_monthly.csv` | US CPI level and inflation measures, Jan 1947 – Dec 2024 |
| `macro_states.py` | Builds the daily inflation states from `us_cpi_monthly.csv` |
| `us_inflation_states_daily.csv` | Daily inflation states, real-time aligned |
| `regime_episodes.py` | Correlation-regime episodes and durations per pair, window (50, 250 days) and labelling variant |
| `regime_episodes_summary.csv`, `regime_reversals.csv` | Its outputs |

## Constructed variables

**Log returns** are `ln(P_t / P_{t-1})` between consecutive trading days. A return is left empty when either day is missing; gaps are never bridged. Bitcoin is taken at its 00:00 UTC close on S&P trading days, so weekend moves fall into Monday's return.

**`ZCB2Y`, `ZCB5Y`, `ZCB10Y`, `ZCB30Y`** are daily log returns of zero-coupon Treasuries from the Gürkaynak–Sack–Wright curve (Campbell, Pflueger and Viceira 2020; David and Veronesi 2013).
- An n-year zero is bought at t−1 and valued at t with maturity n − dt: r_t = n·y_n(t−1) − (n − dt)·y_{n−dt}(t), with y_{n−dt}(t) evaluated exactly from the Svensson parameters of day t.
- Returns span bond-market holidays; gaps longer than 7 calendar days are not bridged.
- **`ZCB10Y` is the bond series of the paper** (from 17 Aug 1971). `ZCB5Y` is a robustness check.

**CPI inflation measures:** `infl_yoy_pct` (12-month change), `infl_mom_ann_pct` (annualised month-on-month log change), `infl_3m_trailing_ann_pct` (3-month **trailing** average), `infl_3m_centred_ann_pct` (3-month **centred** average, months m−1, m, m+1, as in the rules; December 2024 uses the published January 2025 CPI).

**Daily inflation states (`us_inflation_states_daily.csv`).**
- **`inflationHigh_rules` (used by the tests and the simulation)**: the definition of the rule pipeline. The centred 3-month average of the m/m log change of CPI (`infl_cma3`) is above its 2015–2021 75th percentile (3.61% annualised); the state of month m holds on the days of month m. The state is defined ex post: it uses month m+1 and ignores publication lags. It covers 25% of the training days and 49% of the test days.
- Real-time states, used only in the descriptive tables and figures: CPI for month m is used from the first trading day of month m+2. `inflationHigh` enters at 12-month inflation of at least 3.0% and exits below 2.5% (hysteresis); variants `_yoy25`, `_yoy30`, `_yoy40`, `_3m30`, `_p75`, plus `inflationRising` and `inflationOnset`.

## Predicates used by the tests and figures (as in the rules)

- **highVol**: at least 30% of the 50-day window above the 90th percentile of |r| (training period). lowVol: at most 5%.
- **trendDown**: cumulative 50-day log return below −0.02; stressedDown = highVol and trendDown.
- **tailDependent**: at least 2 days in the window on which both series fall below their 5th percentile (training period).
- **Concordance score** (`plot_rule_examples.py`, as in the rules): within a 50-day window, a day is concordant when the two returns have the same sign and their absolute values both rise or both fall relative to the previous day, discordant when they have opposite signs and their absolute values move in the same direction. Positive (negative) correlation when the concordant share exceeds the discordant share (or the reverse) by more than 10 percentage points.
- `regime_episodes.py` and the descriptive figures use the plain **sign** concordance S = 100 × (same-sign − opposite-sign days) / valid days, with a ±10 deadband or hysteresis (enter at |S| ≥ 15, leave below 5).

## Output folders

- **`figures_tables_paper/`** (main text): `fig_rule_examples` (Fig. 2: S&P 500 with gold and with the 10-year zero-coupon Treasury, concordance and the days on which T2 and T1b hold), `fig_stock_bond_50d`, `tab_comovement`, `tab_regimes`, `tab_scale`, `tab_benchmark`.
- **`figures_tables_repository/`**: descriptive figures (`returns_inflation`, `fig_inflation_realtime`, `fig_stock_bond_250d`, `fig_pair_*`), descriptive tables (`tab_series_stats`, `tab_volatility`, `tab_inflation_by_decade`, `tab_regime_episodes`, `tab_regime_reversals`), econometric tests (`tab_econ_tests`, `tab_econ_tests_gold_revised`) and the simulation outputs (`fig_sim_power_econ`, `tab_sim_econ`, `fig_sim_vs_real`, `fig_sim_series_*`).

**Run order** (what `main.py` does): `market_data/build.py` → `market_data/macro_states.py` → `market_data/regime_episodes.py` → `regime_tables.py` → `descriptives.py` → `plot_returns_inflation.py` → `plot_rule_examples.py` → `econometric_tests.py` → simulation (`simulation/README_simulation.ipynb`).

## Econometric benchmarks (`econometric_tests.py`)

Train 2015–2021, test 2022–2024; each template is tested with the standard test of the same phenomenon on daily returns, using the rule's trigger with thresholds from the training period.

| Template | Test |
|---|---|
| T1b | Baur–Lucey regression with GARCH(1,1) errors (nested quantile dummies) |
| T2 | Forbes–Rigobon variance-corrected correlation on tail-dependence days |
| T3b | Correlation on days when both series are calm |
| T4 | Predictive regression with HAC standard errors, at the lag fitted by the rule, slope shift in highVol |
| T7 | Conditional correlation on GARCH-standardised returns, high against low inflation (`inflationHigh_rules`) |

Main results on 2022–2024 (gold of the rule vintage): Forbes–Rigobon finds contagion between the S&P 500 and gold (p = 0.008; p = 0.009 on the revised gold series), but not on 2015–2021; the S&P 500 and Bitcoin have correlation 0.14 on calm days against 0.46 on other days (zero not rejected at 5%, p = 0.06); with the inflation state of the rules, the S&P 500–Bitcoin correlation is higher in high inflation on both periods (p < 0.001 on 2015–2021, p = 0.04 on 2022–2024), while for gold and the bond there is no inflation effect.

## Caveat

Non-synchronous closes: S&P 500 at 16:00 ET, COMEX gold settlement at about 13:30 ET, Bitcoin at 00:00 UTC.